# Encoding Categorical Features

## INFO 442 Group 5 Shuyang Zhang 320230942711

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.base import BaseEstimator, TransformerMixin
tips = pd.read_csv('tips.csv')

## 1. Load seaborn.load_dataset('tips'). Apply OHE to sex, smoker, and day. How many columns does time add with and without drop='first'?

In [19]:
categorical_cols = ['sex', 'smoker', 'day']

ohe_first = OneHotEncoder(drop='first', sparse_output=False)
encoded_first = ohe_first.fit_transform(tips[categorical_cols])
print(f"OHE with drop='first' - columns added: {encoded_first.shape[1]}")

ohe_full = OneHotEncoder(sparse_output=False)
encoded_full = ohe_full.fit_transform(tips[categorical_cols])
print(f"OHE without drop - columns added: {encoded_full.shape[1]}")

time_ohe = OneHotEncoder(sparse_output=False)
time_encoded = time_ohe.fit_transform(tips[['time']])
print(f"time column - columns added: {time_encoded.shape[1]}")

time_ohe_first = OneHotEncoder(drop='first', sparse_output=False)
time_encoded_first = time_ohe_first.fit_transform(tips[['time']])
print(f"time column with drop='first' - columns added: {time_encoded_first.shape[1]}")
print()

OHE with drop='first' - columns added: 5
OHE without drop - columns added: 8
time column - columns added: 2
time column with drop='first' - columns added: 1



answer: With `drop='first'`, the `time` column adds 1 column, since it has 2 unique values: Lunch and Dinner. Without `drop='first'`, it adds 2 columns.

### 2. Apply ordinal encoding to size (treating it as ordered 1–6). Compare R² of a linear regression predicting tip with ordinal vs OHE encoding for size.

In [25]:
X = tips[['size', 'total_bill']].copy()
y = tips['tip']

ohe = OneHotEncoder(drop='first', sparse_output=False)
X_ohe = np.hstack([ohe.fit_transform(tips[['size']]), tips[['total_bill']].values])

ord_enc = OrdinalEncoder()
X_ord = np.hstack([ord_enc.fit_transform(tips[['size']]), tips[['total_bill']].values])

model = LinearRegression()

scores_ohe = cross_val_score(model, X_ohe, y, cv=5, scoring='r2')
scores_ord = cross_val_score(model, X_ord, y, cv=5, scoring='r2')

print(f"One-hot encoding - R²: {scores_ohe.mean():.4f} +/- {scores_ohe.std():.4f}")
print(f"Ordinal encoding - R²: {scores_ord.mean():.4f} +/- {scores_ord.std():.4f}")
print()

One-hot encoding - R²: 0.4135 +/- 0.1415
Ordinal encoding - R²: 0.4657 +/- 0.1225



answer: Ordinal encoding (R² = 0.4657) outperformed one-hot encoding (R² = 0.4135). This is expected because size has a natural order (1 < 2 < 3 < 4 < 5 < 6), which ordinal encoding preserves while one-hot encoding ignores.

### 3. Manually verify that the cross-fold target encoder does not use any target values from the validation fold during encoding (add a print statement inside the loop to inspect the fold sizes).

In [29]:
class TargetEncoderCV(BaseEstimator, TransformerMixin):
    def __init__(self, col, target, n_splits=5, random_state=42):
        self.col = col
        self.target = target
        self.n_splits = n_splits
        self.random_state = random_state
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        X = X.copy()
        global_mean = X[self.target].mean()
        encoded = np.zeros(len(X))
        
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            fold_train = X.iloc[train_idx]
            fold_val = X.iloc[val_idx]
            
            print(f"Fold {fold+1}: Train size={len(train_idx)}, Val size={len(val_idx)}")
            
            means = fold_train.groupby(self.col)[self.target].mean()
            encoded[val_idx] = fold_val[self.col].map(means).fillna(global_mean)
        
        return encoded.reshape(-1, 1)

df_test = pd.DataFrame({
    'category': ['A', 'B', 'C', 'A', 'B', 'C', 'A', 'B', 'C'],
    'target': [1, 2, 3, 4, 5, 6, 7, 8, 9]
})

print("Testing target encoding with debug output:")
encoder = TargetEncoderCV(col='category', target='target', n_splits=3)
result = encoder.fit_transform(df_test, df_test['target'])

Testing target encoding with debug output:
Fold 1: Train size=6, Val size=3
Fold 2: Train size=6, Val size=3
Fold 3: Train size=6, Val size=3


answer: The output shows each fold has 6 training samples and 3 validation samples. The target encoder calculates category means using only the training fold, then applies those means to the validation fold. No target values from the validation fold are used during encoding, confirming no data leakage.

### 4. What happens to handle_unknown='ignore' in OHE when the test set contains a category not seen during training? Demonstrate this with a toy example.

In [38]:
X_train = pd.DataFrame({'city': ['New York', 'London', 'Tokyo']})
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train)
print(f"Training categories: {ohe.categories_[0]}")

X_test = pd.DataFrame({'city': ['New York', 'Paris', 'London', 'Berlin', 'Tokyo']})
X_test_encoded = ohe.transform(X_test)

print("\nTest data encoding (unseen categories -> all zeros):")
result_df = pd.DataFrame(X_test_encoded, columns=ohe.get_feature_names_out())
result_df['original_city'] = X_test['city'].values
print(result_df)

Training categories: ['London' 'New York' 'Tokyo']

Test data encoding (unseen categories -> all zeros):
   city_London  city_New York  city_Tokyo original_city
0          0.0            1.0         0.0      New York
1          0.0            0.0         0.0         Paris
2          1.0            0.0         0.0        London
3          0.0            0.0         0.0        Berlin
4          0.0            0.0         1.0         Tokyo


answer: When handle_unknown='ignore' is set, any unseen category in the test set is encoded as all zeros across all OHE columns. In the example, Paris and Berlin (not present in training) become zero vectors, while known categories (New York, London, Tokyo) are encoded normally.